# Canonical Cells-Clean Quality Checks

This notebook gives visibility into the `cells-clean` canonical dataset.

It checks:
- record count
- identifier format compliance
- NaN leakage
- schema validation status


In [ ]:
from pathlib import Path
import json
import re
import sys
import subprocess

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    # If notebook is opened from inside notebooks/, move to repo root.
    ROOT = ROOT.parent

assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

print('ROOT:', ROOT)


In [ ]:
cells_clean = ROOT / 'src' / 'battinfo' / 'data' / 'examples' / 'cells-clean'
files = sorted(cells_clean.glob('*.json'))
print('cells-clean files:', len(files))
print('sample:', [f.name for f in files[:5]])


In [ ]:
pat = re.compile(r'^https://w3id\.org/battinfo/cell-type/[0-9a-hjkmnp-tv-z]{4}(?:-[0-9a-hjkmnp-tv-z]{4}){3}$')
bad = []
for f in files:
    doc = json.loads(f.read_text(encoding='utf-8'))
    cell_id = (doc.get('cell') or {}).get('id')
    if not isinstance(cell_id, str) or not pat.fullmatch(cell_id):
        bad.append((f.name, cell_id))
print('bad ids:', len(bad))
if bad:
    print('first bad:', bad[0])


In [ ]:
nan_hits = []
for f in files:
    text = f.read_text(encoding='utf-8')
    if '"NaN"' in text or ': NaN' in text:
        nan_hits.append(f.name)
print('files containing NaN tokens:', len(nan_hits))


In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/validate_cells_clean.py'],
    cwd=ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
if result.stderr:
    print(result.stderr)
print('exit code:', result.returncode)


## Debug tip

To normalize bad numeric values in future regenerated data:

```bash
python scripts/clean_cells_clean_nan.py
```
